# Solutions · 00 · What is the method of lines?

Worked solutions to the exercises of [notebook 00](../notebooks/00_what_is_the_method_of_lines.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate as sint
from engmolines import grids, boundary as bc, operators as op, integrators as ti, stencils as st, stiffness
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Replace the central difference in the twenty-line solver by the fourth-order 5-point stencil (interior points only; keep the 3-point stencil next to the boundaries). How does the spatial error change on 40 intervals? And the extreme eigenvalue?

In [2]:
n = 40
x = np.linspace(0, 1, n + 1); dx = x[1] - x[0]
def rhs4(t, u):
    d = np.zeros_like(u)
    d[2:-2] = (-u[4:] + 16 * u[3:-1] - 30 * u[2:-2] + 16 * u[1:-3] - u[:-4]) / (12 * dx**2)   # 5-point, fourth order
    d[1] = (u[2] - 2 * u[1] + u[0]) / dx**2                                                     # 3-point next to the ends
    d[-2] = (u[-1] - 2 * u[-2] + u[-3]) / dx**2
    return d
def rhs2(t, u):
    d = np.zeros_like(u); d[1:-1] = (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2; return d
exact = np.sin(np.pi * x) * np.exp(-np.pi**2 * 0.1)
for name, f in (("3-point", rhs2), ("5-point interior", rhs4)):
    s = sint.solve_ivp(f, (0, 0.1), np.sin(np.pi * x), method="BDF", rtol=1e-10, atol=1e-12)
    lam = np.linalg.eigvals(stiffness.jacobian(f, np.sin(np.pi * x)))
    print(f"{name:<17}: spatial error {np.max(np.abs(s.y[:, -1] - exact)):.2e}; extreme eigenvalue {lam.real.min():.0f}")

3-point          : spatial error 1.89e-04; extreme eigenvalue -6390
5-point interior : spatial error 2.57e-07; extreme eigenvalue -8516


The fourth-order interior stencil cuts the spatial error by almost three orders of magnitude on the same grid, even
with second-order closures next to the boundaries (their local error is small because the solution is flat
there; in general the global error follows the closure order one less, see notebook 01). Its extreme
eigenvalue is −16/(3Δx²) ≈ −8533 instead of −4/Δx² = −6400: a higher-order stencil is somewhat stiffer.

## Exercise 2

Run the twenty-line solver with `method='RK45'` instead of BDF and compare the number of time steps for n = 20, 40, 80. Fit the growth law.

In [3]:
def mol(n):
    x = np.linspace(0, 1, n + 1); dx = x[1] - x[0]
    def f(t, u):
        d = np.zeros_like(u); d[1:-1] = (u[2:] - 2 * u[1:-1] + u[:-2]) / dx**2; return d
    return f, np.sin(np.pi * x)
rows = []
for n in (20, 40, 80, 160):
    f, u0 = mol(n)
    s_rk = sint.solve_ivp(f, (0, 0.1), u0, method="RK45", rtol=1e-6, atol=1e-9)
    s_bdf = sint.solve_ivp(f, (0, 0.1), u0, method="BDF", rtol=1e-6, atol=1e-9)
    rows.append((n, len(s_rk.t) - 1, len(s_bdf.t) - 1))
tab = pd.DataFrame(rows, columns=["n", "RK45 steps", "BDF steps"])
print(tab.to_string(index=False))
print("RK45 step ratio per doubling of n:", np.round(tab["RK45 steps"].to_numpy()[1:] / tab["RK45 steps"].to_numpy()[:-1], 2), " (theory: 4, the n^2 law)")

  n  RK45 steps  BDF steps
 20          39         27
 40         174         27
 80         751         27
160        3075         27
RK45 step ratio per doubling of n: [4.46 4.32 4.09]  (theory: 4, the n^2 law)


RK45's step count quadruples with every doubling of n - the stability limit h < 2.8Δx²/4 shrinks fourfold -
while BDF's stays essentially constant, set by the accuracy of the smooth solution. Explicit cost grows like
n³ (steps × work per step), implicit like n.

## Exercise 3

**Implement it yourself:** write backward Euler for the semi-discrete heat equation using `numpy.linalg.solve` on the dense matrix I - h A, and confirm its first-order time error against `integrators.exponential`.

In [4]:
g = grids.uniform(0, 1, 40)
A = st.diff_operator(g.x, 2, 2).to_dense(); A[0, :] = A[-1, :] = 0.0
u0 = np.sin(np.pi * g.x)
ref = ti.exponential(A, u0, (0, 0.1)).y[-1]
errs = []
for h in (0.01, 0.005, 0.0025):
    M = np.eye(41) - h * A
    u = u0.copy()
    for _ in range(int(round(0.1 / h))):
        u = np.linalg.solve(M, u)                       # backward Euler: (I - hA) u_{n+1} = u_n
    errs.append(np.max(np.abs(u - ref)))
print("backward Euler errors:", ["%.2e" % e for e in errs], "-> ratios", np.round(np.array(errs[:-1]) / np.array(errs[1:]), 2), "(first order: 2)")

backward Euler errors: ['1.74e-02', '8.89e-03', '4.49e-03'] -> ratios [1.96 1.98] (first order: 2)


Backward Euler solves a linear system per step with the fixed matrix I − hA; against the exact semi-discrete
solution its error halves when the step halves. (Factorising I − hA once instead of calling `solve` every
step is what the library's banded Newton does.)